In [1]:

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
from shapely import make_valid, set_precision
from shapely.validation import explain_validity
from shapely.geometry import Polygon


from earthscape.data.gis import gis_to_image, clip_gis_to_boundary
from earthscape.data.downloads import download_zip
# from earthscape.data.kyfromabove import download_data_tiles, get_aoi_index_polygons, mosaic_image_tiles
from earthscape.data.images import image_to_reference_grid, combine_aligned_images, resample_image, filter_image

In [ ]:
'''
1. read gdb layer
    - re-assign map units (if needed)
    - create AOI boundary & save as GeoJSON
2. create GeoTIFFs
    - SG map
    - AOI boundary mask
3. Post-process SG Map
    - fill nodata areas within AOI mask
'''

In [2]:
def create_aoi_polygon(input_path, output_path):
    gdf = gpd.read_file(input_path)
    u = shapely.union_all(gdf.geometry.values, grid_size=0.2)
    if u.geom_type == "MultiPolygon":
        u = max(u.geoms, key=lambda p: p.area)
    aoi = Polygon(u.exterior)
    gdf_aoi = gpd.GeoDataFrame(geometry=[aoi], crs=gdf.crs)
    gdf_aoi.to_file(output_path, driver='GeoJSON')


In [ ]:

# 1. read gdb layer...
gdb_path = r"C:\Users\mamass1\Desktop\ky\KentuckySurficial_v1.gdb"
map_layer = 'MapUnitPolys'
geo_path = r'../data/warren/mask.geojson'
aoi_path = r'../data/warren/boundary.geojson'
map_units = None

# read gdb layer
gdf = gpd.read_file(gdb_path, layer=map_layer)


# re-assign map units 
if not map_units is None:
    for key, val in map_units.items():
        gdf.loc[gdf['Symbol']==key, 'Symbol'] = val


# repair some broken geometries & save map GeoJSON
gdf['geometry'] = gdf['geometry'].buffer(0)
if gdf.is_valid.all():
    gdf.to_file(geo_path, driver='GeoJSON')


# create AOI boundary polygon
create_aoi_polygon(geo_path, aoi_path)


In [4]:
output_resolution = 5

In [6]:
# 2. create GeoTIFFs...

# SG map GeoTIFF
geo_path = r'../data/warren/mask.geojson'
geo_tif_path = geo_path.replace('.geojson', '.tif')
gis_to_image(input_path=geo_path, output_path=geo_tif_path, output_resolution=output_resolution, multiclass_col='Symbol')


# 2B. AOI GeoTIFF
aoi_path = r'../data/warren/boundary.geojson'
aoi_tif_path = aoi_path.replace('.geojson', '.tif')
gis_to_image(input_path=aoi_path, output_path=aoi_tif_path, output_resolution=output_resolution)

In [10]:
import rasterio
from rasterio.plot import show
import numpy as np
from rasterio.mask import mask

In [8]:
with rasterio.open(r'../data/warren/boundary.tif') as src:
    print(src.transform)
    data = src.read(1, masked=True)
    print(np.sum(data.mask))

| 5.00, 0.00, 4665293.10|
| 0.00,-5.00, 3570293.50|
| 0.00, 0.00, 1.00|
138653568


In [9]:
with rasterio.open(r'../data/warren/mask.tif') as src:
    print(src.transform)
    data = src.read(1, masked=True)
    print(np.sum(data.mask))

| 5.00, 0.00, 4665293.14|
| 0.00,-5.00, 3570293.49|
| 0.00, 0.00, 1.00|
138653563


In [11]:
gdf_aoi = gpd.read_file(r'../data/warren/boundary.geojson')

with rasterio.open(r'../data/warren/mask.tif') as src:
    data, _ = mask(src, gdf_aoi.geometry, crop=False, filled=False)

band = data[0]
has_nodata = np.isnan(band.data[~band.mask]).any()

In [12]:
has_nodata

np.False_